# 2.1 Refactorización de P1 (versión final)

Lo elegido en `refactorizado/1.1` pasa a clases con `fit` / `predict`, como scikit-learn, y se compara con `mibici_equipo39/modeling/p1.py`. La versión anterior sigue en `notebooks/2.1-dm-p1-refactorizacion.ipynb`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.preprocessing import TargetEncoder

from mibici_equipo39 import config as C
from mibici_equipo39.features import FEATURE_COLUMNS, eligible_stations, load_tables

metadata, X, y = load_tables()

## Modelos

In [2]:
class SeasonalMeanBaseline:
    levels = (["station_id", "day_of_week", "hour_of_day"], ["station_id", "hour_of_day"])

    def fit(self, frame, y):
        self.means_ = [y.groupby([frame[k] for k in keys], observed=True).mean().rename("mean") for keys in self.levels]
        self.global_ = float(y.mean())
        return self

    def predict(self, frame):
        pred = pd.Series(np.nan, index=frame.index)
        for keys, means in zip(self.levels, self.means_):
            pred = pred.fillna(frame[keys].join(means, on=keys)["mean"])
        return pred.fillna(self.global_).to_numpy()


ENCODED = ["te_station", "te_station_day_hour"]

class StationEncodedHGB:
    def _keys(self, frame):
        station = frame["station_id"].astype(str)
        return pd.DataFrame({"te_station": station,
                             "te_station_day_hour": station + "|" + frame["day_of_week"].astype(str) + "|" + frame["hour_of_day"].astype(str)})

    def fit(self, frame, y):
        self.encoder_ = TargetEncoder(target_type="continuous", cv=5, shuffle=True, random_state=C.SEED)
        encoded = pd.DataFrame(self.encoder_.fit_transform(self._keys(frame), y), columns=ENCODED, index=frame.index)
        self.model_ = HistGradientBoostingRegressor(**C.P1_HGB_PARAMS).fit(pd.concat([frame[FEATURE_COLUMNS], encoded], axis=1), y)
        return self

    def predict(self, frame):
        encoded = pd.DataFrame(self.encoder_.transform(self._keys(frame)), columns=ENCODED, index=frame.index)
        return np.clip(self.model_.predict(pd.concat([frame[FEATURE_COLUMNS], encoded], axis=1)), 0, None)

> **Por qué el encoding va dentro de `fit`.** Así solo ve train, con validación cruzada interna, y `predict` solo transforma. Es imposible ajustarlo por error con validación o prueba.

## Métricas y experimento

In [3]:
SPLITS = {"train": "train", "val": "validation", "test": "test"}

def regression_metrics(y, pred, split):
    out = {}
    for prefix, name in SPLITS.items():
        mask = split.eq(name).to_numpy()
        out[f"{prefix}_mae"] = mean_absolute_error(y[mask], pred[mask])
        out[f"{prefix}_rmse"] = root_mean_squared_error(y[mask], pred[mask])
    return out


class P1Experiment:
    def __init__(self, metadata, X, y):
        keep = metadata["station_id"].isin(eligible_stations(metadata, y)).to_numpy()
        self.metadata = metadata[keep].reset_index(drop=True)
        self.X = X[keep].reset_index(drop=True)
        self.y = y[keep].reset_index(drop=True)
        self.train = self.metadata["split"].eq("train").to_numpy()

    def fit(self):
        frame = pd.concat([self.metadata, self.X], axis=1)
        self.predictions = {
            "baseline": SeasonalMeanBaseline().fit(frame[self.train], self.y[self.train]).predict(frame),
            "candidato": StationEncodedHGB().fit(frame[self.train], self.y[self.train]).predict(frame),
        }
        self.metrics = {k: regression_metrics(self.y.to_numpy(), p, self.metadata["split"]) for k, p in self.predictions.items()}
        self.selected = "candidato" if self.metrics["candidato"]["val_mae"] < self.metrics["baseline"]["val_mae"] else "baseline"
        return self

notebook = P1Experiment(metadata, X, y).fit()
print("seleccionado:", notebook.selected)
pd.DataFrame(notebook.metrics).T[["train_mae", "val_mae", "train_rmse", "val_rmse"]].round(4)

seleccionado: baseline


,train_mae,val_mae,train_rmse,val_rmse
baseline,0.6916,0.7119,1.3751,1.4495
candidato,0.6828,0.7122,1.3844,1.4980


> **Mismos números que la exploración** en validación. Prueba se reporta solo desde MLflow.

## Paso al módulo

| Pieza | Destino |
|---|---|
| `SeasonalMeanBaseline`, `StationEncodedHGB`, `regression_metrics`, `P1Experiment` | `mibici_equipo39/modeling/p1.py` |
| Hiperparámetros y mínimo de salidas | `config.py` |
| Registro en MLflow | `mibici_equipo39/modeling/tracking.py` |
| Controlador común | `mibici_equipo39/modeling/train.py` |

Se ejecuta con `python -m mibici_equipo39.modeling.train --question P1 --member personaN`.

In [4]:
from mibici_equipo39.modeling.p1 import P1Experiment as ModuleExperiment

module = ModuleExperiment(metadata, X, y).fit()
for name in ["baseline", "candidato"]:
    assert np.allclose(notebook.predictions[name], module.predictions[name])
print("notebook y módulo dan las mismas predicciones; seleccionado:", module.selected)

notebook y módulo dan las mismas predicciones; seleccionado: baseline
